# 構造化出力 GP: HigherOrderGP と LatentKroneckerGP

この Notebook では、合成スペクトルデータを使って構造化出力向けの2つの robotorchan モデルを比較します。

- `HigherOrderGP`: 出力を tensor として扱い、分離可能な出力構造を利用します。
- `LatentKroneckerGP`: 設計変数 `X` と、波長・時間・位置などの明示的な出力座標 `T` の積空間をモデル化します。

## 1. これらのモデルを使う場面

`HigherOrderGP` は、各実験から画像・スペクトルグリッド・その他の tensor-valued response が得られ、その tensor shape 自体に意味がある場合に向きます。

`LatentKroneckerGP` は、出力が波長・時間・位置などの座標 `T` で自然に添字付けされ、その軸に沿って明示的に補間・予測したい場合に向きます。

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll, fit_gpytorch_mll_torch
from linear_operator.settings import _fast_solves
from robotorchan.models import HigherOrderGP, LatentKroneckerGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. 合成スペクトルデータ

`X` は2つのプロセス条件を表します。各条件から、波長軸 `T` 上で観測される1本のスペクトルが得られるとします。

In [ ]:
n_train = 12
n_wavelengths = 32
train_X = torch.rand(n_train, 2)
train_T = torch.linspace(0.0, 1.0, n_wavelengths).unsqueeze(-1)
def spectrum(X, T):
    x1 = X[..., 0].unsqueeze(-1)
    x2 = X[..., 1].unsqueeze(-1)
    t = T.squeeze(-1)
    peak1 = torch.exp(-0.5 * ((t - (0.25 + 0.15 * x1)) / 0.07) ** 2)
    peak2 = 0.7 * torch.exp(-0.5 * ((t - (0.70 - 0.12 * x2)) / 0.10) ** 2)
    baseline = 0.15 * x1 + 0.08 * x2
    return peak1 + peak2 + baseline
train_Y = spectrum(train_X, train_T)
train_Y = train_Y + 0.02 * torch.randn_like(train_Y)
print(train_X.shape)
print(train_T.shape)
print(train_Y.shape)

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(min(6, n_train)):
    plt.plot(train_T.squeeze(-1), train_Y[i], alpha=0.8)
plt.xlabel("normalized wavelength")
plt.ylabel("response")
plt.title("Synthetic training spectra")
plt.show()

## 3. HigherOrderGP

HOGP では response tensor をそのまま `train_Y` として渡します。1次元スペクトルも構造化出力ですが、HOGP は画像や時空間グリッドなど、より高階の配列で特に自然です。

学習時には BoTorch の specialized fast solve を有効にし、`fit_gpytorch_mll_torch()` を利用します。

In [ ]:
hogp = HigherOrderGP(train_X=train_X, train_Y=train_Y)
print("supports_mll:", hogp.supports_mll)
print("raw_train_X:", hogp.raw_train_X.shape)
print("raw_train_Y:", hogp.raw_train_Y.shape)
print("raw_train_Yvar:", hogp.raw_train_Yvar)
hogp_mll = hogp.make_mll()
with _fast_solves(True):
    fit_gpytorch_mll_torch(hogp_mll, step_limit=75)
hogp.eval()

In [ ]:
test_X = torch.tensor([[0.65, 0.30]])
hogp_posterior = hogp.posterior(test_X)
hogp_mean = hogp_posterior.mean.squeeze(0).detach()
print("HOGP posterior mean shape:", hogp_mean.shape)

## 4. LatentKroneckerGP

`LatentKroneckerGP` では出力座標を `train_T` として明示的に渡します。そのため robotorchan は共通 raw training tensor に加えて `raw_train_T` も保持します。

このモデルでは効率的な学習・posterior inference のため iterative methods を利用します。

In [ ]:
lkgp = LatentKroneckerGP(train_X=train_X, train_T=train_T, train_Y=train_Y)
print("supports_mll:", lkgp.supports_mll)
print("raw_train_X:", lkgp.raw_train_X.shape)
print("raw_train_T:", lkgp.raw_train_T.shape)
print("raw_train_Y:", lkgp.raw_train_Y.shape)
lkgp_mll = lkgp.make_mll()
with lkgp.use_iterative_methods():
    fit_gpytorch_mll(lkgp_mll)
lkgp.eval()

In [ ]:
with lkgp.use_iterative_methods():
    lkgp_posterior = lkgp.posterior(test_X, train_T)
lkgp_mean = lkgp_posterior.mean.squeeze(0).detach()
lkgp_std = lkgp_posterior.variance.sqrt().squeeze(0).detach()
print("LatentKronecker posterior mean shape:", lkgp_mean.shape)

## 5. 予測結果の比較

合成データなので真のスペクトルも表示できます。実データでは、どの構造化出力仮定が問題に合っているか、また明示的な `T` 軸が有用かどうかがモデル選択の重要な観点です。

In [ ]:
true_y = spectrum(test_X, train_T).squeeze(0)
plt.figure(figsize=(9, 4))
plt.plot(train_T.squeeze(-1), true_y, label="true")
plt.plot(train_T.squeeze(-1), hogp_mean, label="HigherOrderGP")
plt.plot(train_T.squeeze(-1), lkgp_mean, label="LatentKroneckerGP")
plt.fill_between(train_T.squeeze(-1), lkgp_mean - 1.96 * lkgp_std, lkgp_mean + 1.96 * lkgp_std, alpha=0.2, label="LatentKronecker 95% interval")
plt.xlabel("normalized wavelength")
plt.ylabel("response")
plt.legend()
plt.title("Structured-output posterior at one process condition")
plt.show()

## 6. LatentKroneckerGP で新しい T グリッドを評価

出力座標を明示的に持つ利点の1つは、学習時とは異なる波長グリッドでも posterior を評価できることです。

In [ ]:
test_T = torch.linspace(0.05, 0.95, 50).unsqueeze(-1)
with lkgp.use_iterative_methods():
    posterior_new_T = lkgp.posterior(test_X, test_T)
print("new T:", test_T.shape)
print("posterior mean:", posterior_new_T.mean.shape)

## 7. モデル選択

| 状況 | 推奨モデル |
|---|---|
| 画像や多次元グリッドなどの tensor-valued output | `HigherOrderGP` |
| 意味のある座標を持つスペクトル・時系列・空間プロファイル | `LatentKroneckerGP` |
| 新しい出力座標位置で評価したい | `LatentKroneckerGP` |
| 出力 tensor が複数の構造化軸を持つ | `HigherOrderGP` |

物理的に意味のある波長軸を持つスペクトルでは、波長を `T` として明示できる `LatentKroneckerGP` が解釈しやすいことがあります。画像や本当に高階の出力では `HigherOrderGP` がより自然です。

## 8. Bayesian Optimization で使う場合

どちらも構造化出力の surrogate として利用できますが、BO では通常、構造化 response からスカラーまたは低次元の objective を定義する必要があります。例えば、ピーク強度、スペクトル積分面積、target spectrum との距離、波長方向の最大応答、物理知識に基づくスコアなどです。

獲得関数を選ぶ前に、構造化出力から何を最適化するのかを明示してください。